In [ ]:
import argparse
from os import path
import shutil
import torch.optim as optim
import torch.backends.cudnn as cudnn
from time import time
import torch.nn as nn

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
!cp -r /content/gdrive/MyDrive/sources/distill/ /content/;
!cp -r /content/gdrive/MyDrive/sources/models/ /content/;
!cp /content/gdrive/MyDrive/sources/dataset.py /content/dataset.py;
!cp /content/gdrive/MyDrive/sources/utils.py /content/utils.py;
!cp /content/gdrive/MyDrive/wrn40x2/model.pth /content/wrn40x2.pth;

In [ ]:
from dataset import *
from models import *
from distill import *
from utils import *

In [ ]:
from argparse import Namespace

In [ ]:
args = Namespace()
args.data_dir      = "/content/gdrive/MyDrive/"
args.data          = 'CIFAR100'
args.epoch         = 200
args.batch_size    = 64
args.lr            = 0.05
args.momentum      = 0.9
args.lr_decay      = 0.1
args.weight_decay  = 5e-4
args.alpha         = 0.9
args.temperature   = 4
args.model_t       = 'wrn40x2'
args.model         = 'resnet56'
args.beta          = 200
args.qk_dim        = 128
args.trained_dir   = "/content/wrn40x2.pth"
args.schedule      = [150, 180, 210]

In [ ]:
def test_accuracy(model, dataloader, device="cuda"):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
      for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy

In [ ]:
args.guide_layers = LAYER[args.model_t]
args.hint_layers  = LAYER[args.model]

for param in sorted(vars(args).keys()):
      print('--{0} {1}'.format(param, vars(args)[param]))

train_loader, test_loader, args.num_classes, args.image_size = create_loader(args.batch_size, args.data_dir, args.data)

model_t = wrn40x2(num_classes= args.num_classes)
model_t.load_state_dict(torch.load(args.trained_dir ) )

model_s = wrn16x2(num_classes=args.num_classes)

device = torch.device('cuda')
data = torch.randn(2, 3, args.image_size, args.image_size)

model_t.eval()
model_s.eval()

with torch.no_grad():
    feat_t, _ = model_t(data, is_feat=True)
    feat_s, _ = model_s(data, is_feat=True)

module_list = nn.ModuleList([])
module_list.append(model_s)
trainable_list = nn.ModuleList([])

trainable_list.append(model_s)

args.s_shapes = [feat_s[i].size() for i in args.hint_layers]
args.t_shapes = [feat_t[i].size() for i in args.guide_layers]
args.n_t, args.unique_t_shapes = unique_shape(args.t_shapes)

criterion_ce = nn.CrossEntropyLoss()
criterion_kl = DistillKL(args.temperature)
criterion_kd = AFD(args)
module_list.append(criterion_kd)
trainable_list.append(criterion_kd)

criterion = nn.ModuleList([])
criterion.append(criterion_ce)
criterion.append(criterion_kl)
criterion.append(criterion_kd)

optimizer = optim.SGD(trainable_list.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.weight_decay)

module_list.append(model_t)
module_list.cuda()
criterion.cuda()
cudnn.benchmark = True
best_last_acc = 0
Save_file = "s_model.pth"

for epoch in range(1, args.epoch + 1):
    s = time()
    adjust_learning_rate(optimizer, epoch, args)

    train_loss, train_acc1, train_acc5 = train_kl(module_list, optimizer, criterion, train_loader, device, args)
    test_acc1, test_acc5 = test(model_s, test_loader, device)
    if best_last_acc < test_acc1:
        torch.save(model_s.state_dict(), Save_file)
        best_last_acc = test_acc1

    print('Epoch: {0:>3d} |Train Loss: {1:>2.4f} |Train Top1: {2:.4f} |Train Top5: {3:.4f} |Test Top1: {4:.4f} |Test Top5: {5:.4f}| Time: {6:>5.1f} (s)'
          .format(epoch, train_loss, train_acc1, train_acc5, test_acc1, test_acc5, time() - s))

acc_t = test_accuracy(model_t, test_loader)
print("Teacher Accuracy on test data:",acc_t)
acc_s = test_accuracy(model_s, test_loader)
print("Student Accuracy on test data:",acc_t)


--alpha 0.9
--batch_size 64
--beta 200
--data CIFAR100
--data_dir /content/gdrive/MyDrive/
--epoch 200
--guide_layers [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
--hint_layers [1 2 3 4 5 6]
--lr 0.05
--lr_decay 0.1
--model wrn16x2
--model_t wrn40x2
--momentum 0.9
--qk_dim 128
--schedule [150, 180, 210]
--temperature 4
--trained_dir /content/wrn40x2.pth
--weight_decay 0.0005
Epoch:   1 |Train Loss: 7.3477 |Train Top1: 0.1227 |Train Top5: 0.3481 |Test Top1: 0.1743 |Test Top5: 0.4579| Time:  57.5 (s)
Epoch:   2 |Train Loss: 5.6506 |Train Top1: 0.2778 |Train Top5: 0.5991 |Test Top1: 0.3004 |Test Top5: 0.6280| Time:  56.6 (s)
Epoch:   3 |Train Loss: 4.8651 |Train Top1: 0.3872 |Train Top5: 0.7278 |Test Top1: 0.3636 |Test Top5: 0.7074| Time:  57.0 (s)
Epoch:   4 |Train Loss: 4.4647 |Train Top1: 0.4505 |Train Top5: 0.7820 |Test Top1: 0.4475 |Test Top5: 0.7626| Time:  55.4 (s)
Epoch:   5 |Train Loss: 4.2376 |Train Top1: 0.4906 |Train Top5: 0.8126 |Test Top1: 0.4178 |Test Top5: 0.759